In [1]:
import os
import torch
from diffusers import AutoencoderKL
# Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


/Users/fdb/Experiments/latent-diffusion-from-scratch/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [2]:
SNAPSHOT_PATH = "output/snapshots"
os.makedirs(SNAPSHOT_PATH, exist_ok=True)

In [3]:
# 1. Load Pre-trained VAE (The "Compressor")
# We use standard Stable Diffusion VAE weights to compress 512px -> 64px latents
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
vae.requires_grad_(False) # Freeze VAE
print()

In [4]:
# %%
# Export Corrected VAE Decoder (Wrapper method)
print("Exporting Corrected VAE Decoder...")

# --- 1. Define the Wrapper Class ---
# This class combines the post_quant_conv layer and the decoder into a single module.
class VaeDecoderWrapper(torch.nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.post_quant_conv = vae.post_quant_conv
        self.decoder = vae.decoder

    def forward(self, z):
        # Apply the quantization convolution first (CRITICAL FIX)
        z = self.post_quant_conv(z)
        # Then decode
        return self.decoder(z)

# --- 2. Instantiate the Wrapper ---
vae.eval()
vae_wrapper = VaeDecoderWrapper(vae).to(device)

# --- 3. Create Dummy Input ---
# Shape: (Batch=1, Channels=4, Height=64, Width=64)
dummy_latent_input = torch.randn(1, 4, 64, 64).to(device)

# --- 4. Define Output Path ---
vae_onnx_path = f"{SNAPSHOT_PATH}/vae_decoder.onnx"

# --- 5. Export ---
torch.onnx.export(
    vae_wrapper,           
    dummy_latent_input,
    vae_onnx_path,
    export_params=True,
    opset_version=14,      
    dynamo=False,
    do_constant_folding=True,
    input_names=["latent_sample"],
    output_names=["sample"],
    dynamic_axes={
        "latent_sample": {0: "batch_size"},
        "sample": {0: "batch_size"}
    },
)
print(f"VAE Decoder exported to {vae_onnx_path}")

# %%

Exporting Corrected VAE Decoder...


/var/folders/5b/q4chvlgs3dn2myf9tj7v_kzh0000gn/T/ipykernel_21823/1482748954.py:31: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/Users/fdb/Experiments/latent-diffusion-from-scratch/.venv/lib/python3.12/site-packages/diffusers/models/upsampling.py:147: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the 

VAE Decoder exported to output/snapshots/vae_decoder.onnx
